In [ ]:
import pandas as pd

In [203]:
clean_df = pd.read_csv(
    "../data/processed/MFG10YearTerminationData_clean.csv"
)

clean_df.head(10)

,EmployeeID,recorddate_key,birthdate_key,orighiredate_key,terminationdate_key,age,length_of_service,city_name,department_name,job_title,store_name,gender_short,gender_full,termreason_desc,termtype_desc,STATUS_YEAR,STATUS,BUSINESS_UNIT,date_conflict_flag
0,1318,2006-12-31,1954-01-03,1989-08-28,NaN,52,17,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2006,ACTIVE,HEADOFFICE,False
1,1318,2007-12-31,1954-01-03,1989-08-28,NaN,53,18,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2007,ACTIVE,HEADOFFICE,False
2,1318,2008-12-31,1954-01-03,1989-08-28,NaN,54,19,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2008,ACTIVE,HEADOFFICE,False
3,1318,2009-12-31,1954-01-03,1989-08-28,NaN,55,20,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2009,ACTIVE,HEADOFFICE,False
4,1318,2010-12-31,1954-01-03,1989-08-28,NaN,56,21,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2010,ACTIVE,HEADOFFICE,False
5,1318,2011-12-31,1954-01-03,1989-08-28,NaN,57,22,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2011,ACTIVE,HEADOFFICE,False
6,1318,2012-12-31,1954-01-03,1989-08-28,NaN,58,23,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2012,ACTIVE,HEADOFFICE,False
7,1318,2013-12-31,1954-01-03,1989-08-28,NaN,59,24,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2013,ACTIVE,HEADOFFICE,False
8,1318,2014-12-31,1954-01-03,1989-08-28,NaN,60,25,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2014,ACTIVE,HEADOFFICE,False
9,1318,2015-12-31,1954-01-03,1989-08-28,NaN,61,26,Vancouver,Executive,CEO,35,M,Male,Not Applicable,Not Applicable,2015,ACTIVE,HEADOFFICE,False


In [204]:
# SAP tarzı kişisel bilgiler tablosu için çalışan numarası,
# doğum tarihi ve cinsiyeti seçiyoruz.
# Aynı bilgileri içeren yıllık tekrarları tekilleştiriyoruz.
# Ana veri değişmiyor; her çalışan için tek satırlık ayrı tablo oluşuyor.
personal_data = clean_df[
    ["EmployeeID", "birthdate_key", "gender_short"]
].drop_duplicates() #3 bilgisi aynı olan kayıtları bir kez tuttuk.

personal_data.shape

(6284, 3)

In [205]:
personal_data = personal_data.rename(columns={
    "EmployeeID": "PERNR",
    "birthdate_key": "GBDAT",
    "gender_short": "GESCH"
})

personal_data.head()

,PERNR,GBDAT,GESCH
0,1318,1954-01-03,M
10,1319,1957-01-03,F
20,1320,1955-01-02,F
30,1321,1959-01-02,M
40,1322,1958-01-09,M


In [206]:
personal_data["GESCH"] = personal_data["GESCH"].replace({
    "M": 1,
    "F": 2
})

personal_data.head()

<yerel proje klasörü>
  personal_data["GESCH"] = personal_data["GESCH"].replace({


,PERNR,GBDAT,GESCH
0,1318,1954-01-03,1
10,1319,1957-01-03,2
20,1320,1955-01-02,2
30,1321,1959-01-02,1
40,1322,1958-01-09,1


In [207]:
print("Boş cinsiyet:", personal_data["GESCH"].isna().sum())
print("Tekrarlanan çalışan:", personal_data["PERNR"].duplicated().sum())

Boş cinsiyet: 0
Tekrarlanan çalışan: 0


In [208]:
#oluşturduğumuz tabloyu CSV'ye kaydedelim.
personal_data.to_csv(
    "../data/processed/PA0002_personal_data.csv",
    index=False,
    encoding="utf-8-sig"
)

In [209]:
#organizasyon tablosunu oluşturuyoruz
#Çalışanın hangi tarihte hangi departman,
#görev, mağaza ve şehirde bulunduğunu tutacak.
org_data = clean_df[[
    "EmployeeID",
    "recorddate_key",
    "department_name",
    "job_title",
    "store_name",
    "city_name",
    "BUSINESS_UNIT"
]].copy()

org_data.shape

(49653, 7)

In [210]:
#organizasyon tablosundaki sütun adlarını SAP tarzı adlara çevirelim
org_data = org_data.rename(columns={
    "EmployeeID": "PERNR",
    "recorddate_key": "BEGDA",
    "department_name": "ORGEH_TXT",
    "job_title": "PLANS",
    "store_name": "WERKS",
    "city_name": "ORT01",
    "BUSINESS_UNIT": "GSBER"
})

org_data.head()

,PERNR,BEGDA,ORGEH_TXT,PLANS,WERKS,ORT01,GSBER
0,1318,2006-12-31,Executive,CEO,35,Vancouver,HEADOFFICE
1,1318,2007-12-31,Executive,CEO,35,Vancouver,HEADOFFICE
2,1318,2008-12-31,Executive,CEO,35,Vancouver,HEADOFFICE
3,1318,2009-12-31,Executive,CEO,35,Vancouver,HEADOFFICE
4,1318,2010-12-31,Executive,CEO,35,Vancouver,HEADOFFICE


In [211]:
print("Satır sayısı:", len(org_data))
print("Boş değerler:")
print(org_data.isna().sum())

Satır sayısı: 49653
Boş değerler:
PERNR        0
BEGDA        0
ORGEH_TXT    0
PLANS        0
WERKS        0
ORT01        0
GSBER        0
dtype: int64


In [212]:
org_data.to_csv(
    "../data/processed/PA0001_org_assignment.csv",
    index=False,
    encoding="utf-8-sig"
)

In [272]:
#01-işe giriş kodu
hire_actions = clean_df[
    ["EmployeeID", "orighiredate_key"]
].drop_duplicates()

hire_actions = hire_actions.rename(columns={
    "EmployeeID": "PERNR",
    "orighiredate_key": "BEGDA"
})

hire_actions["MASSN"] = "01"

hire_actions.head()

,PERNR,BEGDA,MASSN
0,1318,1989-08-28,01
10,1319,1989-08-28,01
20,1320,1989-08-28,01
30,1321,1989-08-28,01
40,1322,1989-08-31,01


In [273]:
#ayrılma olaylarını inceliyoruz
exit_actions = clean_df[
    clean_df["STATUS"] == "TERMINATED"
][[
    "EmployeeID",
    "terminationdate_key",
    "termreason_desc",
    "termtype_desc"
]].copy()

exit_actions.shape

(1485, 4)

In [215]:
#"10" işten ayrılma kodu
exit_actions = exit_actions.rename(columns={
    "EmployeeID": "PERNR",
    "terminationdate_key": "BEGDA",
    "termreason_desc": "MASSG",
    "termtype_desc": "TERMTYPE"
})

exit_actions["MASSN"] = "10"

exit_actions.head()

,PERNR,BEGDA,MASSG,TERMTYPE,MASSN
48168,3987,2010-09-03,Retirement,Voluntary,10
48169,4107,2011-01-12,Retirement,Voluntary,10
48170,4347,2006-09-23,Retirement,Voluntary,10
48171,4263,2011-06-22,Retirement,Voluntary,10
48172,4707,2012-10-30,Retirement,Voluntary,10


In [216]:
#işe alım ile ayrılma olaylarını birleştiriyoruz
actions = pd.concat(
    [hire_actions, exit_actions],
    ignore_index=True
)

actions.shape

(7769, 5)

In [217]:
actions["MASSN"].value_counts()

MASSN
01    6284
10    1485
Name: count, dtype: int64

In [218]:
actions.to_csv(
    "../data/processed/PA0000_actions.csv",
    index=False,
    encoding="utf-8-sig"
)

In [219]:
from pathlib import Path

for filename in [
    "PA0000_actions.csv",
    "PA0001_org_assignment.csv",
    "PA0002_personal_data.csv"
]:
    print(filename, Path("../data/processed", filename).exists())

PA0000_actions.csv True
PA0001_org_assignment.csv True
PA0002_personal_data.csv True


In [220]:
employee_year = clean_df[[
    "EmployeeID",
    "STATUS_YEAR",
    "recorddate_key",
    "age",
    "length_of_service",
    "STATUS",
    "department_name",
    "job_title",
    "store_name",
    "city_name",
    "BUSINESS_UNIT",
    "date_conflict_flag"
]].copy()

employee_year.shape

(49653, 12)

In [221]:
employee_year.to_csv(
    "../data/processed/employee_year_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)